# Tuning

Find the optimal `n_clusters` and `n_segments` for your data.
Unlike tsam's built-in tuning, this evaluates each candidate across **all slices**,
so the result is optimal for your full multi-dimensional dataset.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook_connected"

da = sample_energy_data(n_days=30)
print(f"Shape: {dict(da.sizes)}")

## Grid search

`find_best_combination` tests all `(n_clusters, n_segments)` pairs up to a maximum
number of timesteps. The full unfiltered history is preserved — useful for
heatmaps and exploration.

In [ ]:
grid = tsam_xarray.find_best_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    max_timesteps=48,
    show_progress=False,
)
print(f"Tested {len(grid.history)} combinations")
print(f"Best: n_clusters={grid.n_clusters}, n_segments={grid.n_segments}")
print(f"Overall RMSE: {grid.rmse:.4f}")

## Summary matrix

The RMSE for each tested `(n_clusters, n_segments)` combination as an xarray Dataset.

In [ ]:
grid.summary_matrix

## RMSE heatmap

In [ ]:
grid.summary_matrix["rmse"].plotly.imshow(
    x="n_segments",
    y="n_clusters",
    title="RMSE by (n_clusters, n_segments)",
)

## Pareto front

`find_pareto_front` runs the same grid search but filters to non-dominated
configurations — those where no other combo has both lower RMSE *and* fewer timesteps.

In [ ]:
pareto = tsam_xarray.find_pareto_front(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    max_timesteps=48,
    show_progress=False,
)
print(f"Pareto-optimal: {len(pareto.history)} of {len(grid.history)} tested")
pareto.summary

## Best result

The best `AggregationResult` is ready to use — evaluated across all slices.

In [ ]:
grid.best_result.reconstructed.plotly.line(
    x="time", facet_col="scenario", animation_frame="region"
)

In [ ]:
grid.best_result.cluster_representatives.sel(
    scenario="low",
    variable="solar",
).plotly.line(
    line_shape="hv",
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Best typical periods (solar, low scenario)",
)

## Find optimal for a specific data reduction

`find_optimal_combination` tests only the boundary combos for a target reduction — faster.

In [ ]:
result_opt = tsam_xarray.find_optimal_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    data_reduction=0.05,
    show_progress=False,
)
print(f"Best for 5% reduction: {result_opt.n_clusters}c x {result_opt.n_segments}s")
print(f"RMSE: {result_opt.rmse:.4f}")
result_opt.summary